# Project data cleaning validation

This notebook reruns the inspection and cleaning workflow from the unchanged files in `data/raw/`. It displays the evidence used for the 14-item checklist.

## Project question

Which occupations may deserve priority for AI-transition training and reskilling support based on observed AI exposure, task penetration, employment, wages, and projected employment change?

Raw source files are preserved unchanged. Cleaned tables and evidence reports are written to `outputs/`.

In [7]:
from pathlib import Path
import contextlib
import io
import runpy
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS = ROOT / 'outputs'
print(ROOT)

/Users/emerald/Desktop/dso576/DSO576-Project-


In [8]:
# Restart-and-run check: these commands regenerate the evidence from raw data.
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path(str(ROOT / 'code' / 'inspect_raw_data.py'), run_name='__main__')
    runpy.run_path(str(ROOT / 'code' / 'clean_data.py'), run_name='__main__')

In [9]:
row_counts = pd.read_csv(OUTPUTS / 'cleaning_row_counts.csv')
row_counts

,table,rows_before,rows_after,row_change,columns_after,exact_duplicates_before,exact_duplicates_after
0,job_exposure,756,756,0,4,0,0
1,task_penetration,17998,17991,-7,4,6,0
2,onet_occupation_data,1016,1016,0,4,0,0
3,onet_task_statements,18838,18838,0,10,0,0
4,bls_table_1_2,1112,1112,0,16,0,0


In [10]:
missing = pd.read_csv(OUTPUTS / 'cleaning_missing_values.csv')
missing

,table,column,missing_count,missing_percent
0,onet_task_statements,Task Type,418,2.219
1,onet_task_statements,Incumbents Responding,587,3.116
2,bls_table_1_2,"Median annual wage, dollars, 2025",9,0.809
3,bls_table_1_2,Work experience in a related occupation,716,64.388
4,bls_table_1_2,Typical on-the-job training needed to attain c...,321,28.867


In [11]:
print('Duplicate checks')
display(pd.read_csv(OUTPUTS / 'cleaning_duplicates.csv'))
print('Conversion checks')
display(pd.read_csv(OUTPUTS / 'cleaning_conversions.csv'))
print('Merge checks')
display(pd.read_csv(OUTPUTS / 'cleaning_merges.csv'))
print('Unmatched records')
display(pd.read_csv(OUTPUTS / 'unmatched_exposure_records.csv'))

Duplicate checks


,table,duplicate_type,count,conflicting_task_groups
0,task_penetration,normalized_task_and_penetration,7,0


Conversion checks


,column,nonmissing_before,nonmissing_after,failed_conversions
0,penetration,17998,17998,0
1,Incumbents Responding,18251,18251,0
2,"Employment, 2025",1112,1112,0
3,"Employment, 2035",1112,1112,0
4,"Employment distribution, percent, 2025",1112,1112,0
5,"Employment distribution, percent, 2035",1112,1112,0
6,"Employment change, numeric, 2025–35",1112,1112,0
7,"Employment change, percent, 2025–35",1112,1112,0
8,"Occupational openings, 2025–35 annual average",1112,1112,0
9,"Median annual wage, dollars, 2025",1103,1103,0


Merge checks


,merge,left_rows,right_rows,output_rows,matched,left_only,right_only
0,job_exposure to BLS line items,756,831,756,755,1,0


Unmatched records


,occ_code,title,observed_exposure,soc6,2025 National Employment Matrix title,2025 National Employment Matrix code,Occupation type,"Employment, 2025","Employment, 2035","Employment distribution, percent, 2025","Employment distribution, percent, 2035","Employment change, numeric, 2025–35","Employment change, percent, 2025–35","Percent self-employed, 2025","Occupational openings, 2025–35 annual average","Median annual wage, dollars, 2025",Typical education needed for entry,Work experience in a related occupation,Typical on-the-job training needed to attain competency in the occupation
0,11-1031,Legislators,0.0,11-1031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
record_checks = pd.read_csv(OUTPUTS / 'record_checks.csv')
record_checks

,check_id,case,original,expected,actual,matches_expected
0,R1,unmatched occupation,11-1031 | Legislators,No BLS line-item match; preserve and flag the ...,BLS matches = 0,True
1,R2,duplicate task penetration,Write interesting and effective press releases...,One row retained for the normalized task and p...,Retained rows = 1,True
2,R3,missing BLS wage,27-2011 | —,"Formatted missing marker becomes missing, not ...",Missing,True
3,R4,missing task type,15267 | nan,Missing task type remains missing; do not infe...,Missing,True
4,R5,successful occupation merge,11-1011 | Chief Executives | exposure 0.0333,One BLS line-item match with a numeric wage,Matches = 1; wage = 213990,True


## Checklist evidence notes

1. Vision and question: stated above.
2. Row definitions: occupation tables use SOC keys; task statements use `Task ID`; task penetration uses normalized task text plus penetration.
3. Source preservation: raw files remain in `data/raw/` and are never overwritten.
4. Types: formatted BLS numeric fields and task respondent counts are converted with failed conversions reported.
5. Text: whitespace and case-normalized matching keys are added while source text is retained.
6. Duplicates: exact and normalized duplicate counts are reported.
7. Missingness: before/after missing values are reported; missing values are not imputed.
8. Suspicious values: missing wage markers and unmatched occupations are flagged for review.
9. Merges: the exposure-to-BLS merge reports matched and unmatched keys.
10. Reconciliation: row counts before and after cleaning are in `cleaning_row_counts.csv`.
11. Calculations: numeric conversion and missing-value evidence are available for metric denominators.
12. Individual records: five checks are in `record_checks.csv`.
13. Reproducibility: restart the kernel and run all cells from the raw files.
14. Limitations: one exposure occupation lacks a BLS line-item match; missing source fields remain unresolved.